In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

survival_df = pd.read_csv("TCGA_data/survival_data.txt", sep="\t")
phenotype_df = pd.read_csv("TCGA_data/survival_full_set.csv")

merged_df = survival_df.merge(
    phenotype_df,
    on="sample",
    how="inner"
)

# Split by survival status
survived = merged_df[merged_df['OS'] == 1]
not_survived = merged_df[merged_df['OS'] == 0]

RT_df = merged_df[merged_df['treatment_or_therapy.treatments.diagnoses'] == "RT"]
TMZ_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "TMZ"]
RT_and_TMZ_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "RT/TMZ"]
no_treatment_df = merged_df[merged_df["treatment_or_therapy.treatments.diagnoses"] == "no treatment"]

# --- Event encoding ---
merged_df["event"] = merged_df["vital_status.demographic"].map({
    "Dead": 1,
    "Alive": 0
})

# --- Clean OS.time ---
merged_df["OS.time"] = pd.to_numeric(merged_df["OS.time"], errors="coerce")

# --- Drop missing required fields ---
merged_df = merged_df.dropna(subset=["OS.time", "event", "Methylation_Status"])

# --- TMZ group classification ---
def classify_tmz(val):
    val_str = str(val).lower()
    if "tmz" in val_str or "temozolomide" in val_str:
        return "TMZ"        # catches both "TMZ" and "RT/TMZ"
    else:
        return "Non-TMZ"    # catches RT alone, no treatment, etc.

merged_df["TMZ_group"] = merged_df[
    "treatment_or_therapy.treatments.diagnoses"
].apply(classify_tmz)

output_path_2 = 'TCGA_data/merged_data.csv'
merged_df.to_csv(output_path_2, index=False)

In [3]:
# ── RNA-seq Data Load and Merge ──────────────────────────────────────────────

# --- Load RNA-seq data ---
rna_df = pd.read_csv("TCGA_data/RNAseq_data.txt", sep="\t", index_col="Ensembl_ID")

# --- Strip version numbers from Ensembl IDs ---
rna_df.index = rna_df.index.str.split(".").str[0]

print("RNA-seq shape:", rna_df.shape)

# --- Key GBM genes and their Ensembl IDs (no version) ---
genes_of_interest = {
    "ENSG00000170430": "MGMT",
    "ENSG00000128052": "IDH1",
    "ENSG00000182054": "IDH2",
    "ENSG00000146648": "EGFR",
    "ENSG00000171862": "PTEN",
    "ENSG00000141510": "TP53",
    "ENSG00000164362": "TERT"
}

# --- Filter for genes of interest ---
available_genes = {k: v for k, v in genes_of_interest.items() 
                   if k in rna_df.index}

print("\nGenes found after stripping versions:")
print(available_genes)

# --- Subset and transpose ---
rna_subset = rna_df.loc[available_genes.keys()].T
rna_subset.columns = [available_genes[col] for col in rna_subset.columns]
rna_subset.index.name = "sample"
rna_subset = rna_subset.reset_index()

# --- Check sample ID format in both dataframes ---
print("\nRNA sample IDs (first 5):")
print(rna_subset["sample"].head().tolist())
print("\nmerged_df sample IDs (first 5):")
print(merged_df["sample"].head().tolist())

# --- Trim RNA sample IDs to match merged_df format ---
rna_subset["sample"] = rna_subset["sample"].str[:15]

print("\nRNA subset shape:", rna_subset.shape)
print(rna_subset.head())

# --- Reload merged_df fresh before merging to avoid column loss ---
merged_df = merged_df.merge(rna_subset, on="sample", how="inner")

print("\nMerged shape after RNA join:", merged_df.shape)
print("\nGenes available:")
found_genes = [g for g in available_genes.values() if g in merged_df.columns]
print(found_genes)
print(merged_df[found_genes].describe())

RNA-seq shape: (60660, 175)

Genes found after stripping versions:
{'ENSG00000170430': 'MGMT', 'ENSG00000128052': 'IDH1', 'ENSG00000182054': 'IDH2', 'ENSG00000146648': 'EGFR', 'ENSG00000171862': 'PTEN', 'ENSG00000141510': 'TP53', 'ENSG00000164362': 'TERT'}

RNA sample IDs (first 5):
['TCGA-27-2521-01A', 'TCGA-19-1390-01A', 'TCGA-27-1830-01A', 'TCGA-32-1970-01A', 'TCGA-06-0190-01A']

merged_df sample IDs (first 5):
['TCGA-14-1043-01B', 'TCGA-14-0781-01B', 'TCGA-32-1980-01A', 'TCGA-14-1395-01B', 'TCGA-4W-AA9S-01A']

RNA subset shape: (175, 8)
            sample      MGMT      IDH1      IDH2      EGFR      PTEN  \
0  TCGA-27-2521-01  0.610983  2.113134  6.377873  4.852713  2.807932   
1  TCGA-19-1390-01  0.408277  1.505484  6.233779  2.140975  3.266022   
2  TCGA-27-1830-01  1.217231  2.821486  5.585350  3.773406  3.458290   
3  TCGA-32-1970-01  1.494262  2.871469  5.859010  9.055941  2.828368   
4  TCGA-06-0190-01  1.821220  2.694457  5.949563  3.035096  3.187863   

       TP53      TER

In [4]:
# --- Step 1: Rebuild merged_df from scratch ---
survival_df = pd.read_csv("TCGA_data/survival_data.txt", sep="\t")
phenotype_df = pd.read_csv("TCGA_data/survival_full_set.csv")

merged_df = survival_df.merge(phenotype_df, on="sample", how="inner")

print("Fresh merged_df shape:", merged_df.shape)
print("Sample IDs (first 5):", merged_df["sample"].head().tolist())

Fresh merged_df shape: (599, 75)
Sample IDs (first 5): ['TCGA-12-0657-01A', 'TCGA-32-1977-01A', 'TCGA-19-1791-01A', 'TCGA-28-1757-01A', 'TCGA-19-2624-01A']


In [5]:
# --- Step 2: Trim merged_df sample IDs to 15 characters ---
merged_df["sample"] = merged_df["sample"].str[:15]

print("merged_df sample IDs (first 5):", merged_df["sample"].head().tolist())
print("RNA sample IDs (first 5):", rna_subset["sample"].head().tolist())

# --- Step 3: Check overlap before merging ---
overlap = set(merged_df["sample"]).intersection(set(rna_subset["sample"]))
print(f"\nOverlapping samples: {len(overlap)}")

# --- Step 4: Merge ---
merged_df = merged_df.merge(rna_subset, on="sample", how="inner")

print("\nMerged shape after RNA join:", merged_df.shape)
print("\nGene columns summary:")
genes = ["MGMT", "IDH1", "IDH2", "EGFR", "PTEN", "TP53", "TERT"]
print(merged_df[genes].describe())

merged_df sample IDs (first 5): ['TCGA-12-0657-01', 'TCGA-32-1977-01', 'TCGA-19-1791-01', 'TCGA-28-1757-01', 'TCGA-19-2624-01']
RNA sample IDs (first 5): ['TCGA-27-2521-01', 'TCGA-19-1390-01', 'TCGA-27-1830-01', 'TCGA-32-1970-01', 'TCGA-06-0190-01']

Overlapping samples: 155

Merged shape after RNA join: (160, 82)

Gene columns summary:
             MGMT        IDH1        IDH2        EGFR        PTEN        TP53  \
count  160.000000  160.000000  160.000000  160.000000  160.000000  160.000000   
mean     1.393800    2.417817    5.910396    5.874403    2.673972    4.070532   
std      0.612576    0.582550    0.445076    2.096836    0.452955    0.590101   
min      0.373509    0.961697    4.748128    2.140975    1.441643    1.962105   
25%      0.885921    2.112984    5.665929    4.067586    2.427089    3.736002   
50%      1.457061    2.415906    5.909274    5.928023    2.729052    4.090549   
75%      1.786860    2.752373    6.246034    7.639079    2.978536    4.410592   
max      2.82